In [142]:
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads

In [143]:
census_df = pd.read_csv('CensusTractsTIGER2010.csv')
census_df['geometry'] = census_df['the_geom'].apply(loads)

census_gdf = gpd.GeoDataFrame(census_df, geometry='geometry')
census_gdf.set_crs(epsg=4326, inplace=True)

census_gdf = census_gdf.to_crs(epsg=3857) 

In [144]:
# Função para calcular o centro das regiões
def calcular_centros(census_gdf):
   
    census_gdf['centro'] = census_gdf.geometry.centroid
    return census_gdf[['GEOID', 'centro']]

In [145]:
# Calculando os centros
centros_df = calcular_centros(census_gdf)
centros_gdf = gpd.GeoDataFrame(centros_df, geometry='centro')

census_gdf

,the_geom,STATEFP10,COUNTYFP10,TRACTCE10,GEOID,NAME10,NAMELSAD10,COMMAREA,COMMAREA_N,NOTES,geometry,centro
0,MULTIPOLYGON (((-87.62404799998049 41.73021699...,17,31,842400,17031842400,8424.0,Census Tract 8424,44,44,NaN,"MULTIPOLYGON (((-9754264.405 5120652.536, -975...",POINT (-9754931.488 5122481.305)
1,MULTIPOLYGON (((-87.6860799999848 41.822956000...,17,31,840300,17031840300,8403.0,Census Tract 8403,59,59,NaN,"MULTIPOLYGON (((-9761169.775 5134495.894, -976...",POINT (-9760702.448 5135861.124)
2,MULTIPOLYGON (((-87.62934700001182 41.85279700...,17,31,841100,17031841100,8411.0,Census Tract 8411,34,34,NaN,"MULTIPOLYGON (((-9754854.287 5138954.59, -9754...",POINT (-9755494.457 5138686.94)
3,MULTIPOLYGON (((-87.68813499997718 41.85569099...,17,31,841200,17031841200,8412.0,Census Tract 8412,31,31,NaN,"MULTIPOLYGON (((-9761398.537 5139387.107, -976...",POINT (-9760864.979 5139367.823)
4,MULTIPOLYGON (((-87.63312200003458 41.87448800...,17,31,839000,17031839000,8390.0,Census Tract 8390,32,32,NaN,"MULTIPOLYGON (((-9755274.518 5142196.858, -975...",POINT (-9755083.33 5141682.035)
...,...,...,...,...,...,...,...,...,...,...,...,...
796,MULTIPOLYGON (((-87.65745700003984 41.93257799...,17,31,70400,17031070400,704.0,Census Tract 704,7,7,NaN,"MULTIPOLYGON (((-9757983.477 5150885.3, -97579...",POINT (-9757838.681 5150345.031)
797,MULTIPOLYGON (((-87.66349399996524 41.93036099...,17,31,70500,17031070500,705.0,Census Tract 705,7,7,NaN,"MULTIPOLYGON (((-9758655.513 5150553.561, -975...",POINT (-9758365.885 5150341.781)
798,MULTIPOLYGON (((-87.71436299999318 41.98299699...,17,31,130300,17031130300,1303.0,Census Tract 1303,13,13,NaN,"MULTIPOLYGON (((-9764318.224 5158432.817, -976...",POINT (-9764210.076 5158966.272)
799,MULTIPOLYGON (((-87.71317299997403 41.85523099...,17,31,292200,17031292200,2922.0,Census Tract 2922,29,29,NaN,"MULTIPOLYGON (((-9764185.754 5139318.358, -976...",POINT (-9764467.809 5139530.711)


In [146]:
crimes_df = pd.read_csv("crime-number-central-area.csv")

census_centros_gdf = centros_gdf.merge(crimes_df, on='GEOID', how='left')

quartis = census_centros_gdf['crime_number'].quantile([0.25, 0.5, 0.75, 1]).to_dict()

In [147]:
#import pandas as pd
#from shapely.geometry import Point
#
## Função para calcular a proporção de centros dentro de raios e faixas de valores
#def calcular_heterofilia(census_centros_gdf, raios, faixas):
#    resultados = []
#
#    for raio in raios:
#        for faixa, (limite_inferior, limite_superior) in faixas.items():
#            for index, row in census_centros_gdf.iterrows():
#                centro_id = row['GEOID']
#                centro_referencia = row['centro']
#                crime_count = row['crime_number']
#
#                circulo = centro_referencia.buffer(raio)
#
#                centros_dentro_raio = census_centros_gdf[census_centros_gdf.geometry.intersects(circulo)]
#
#                total_dentro_raio = len(centros_dentro_raio)
#
#                if total_dentro_raio > 0:
#                    # Verificar se o crime_count está dentro dos limites da faixa
#                    if limite_inferior <= crime_count <= limite_superior:
#                        contagem_faixa = ((centros_dentro_raio['crime_number'] >= limite_inferior) & 
#                                          (centros_dentro_raio['crime_number'] <= limite_superior)).sum()
#                        proporcao = (contagem_faixa / total_dentro_raio)
#
#                        resultados.append({
#                            'centro_id': centro_id,
#                            'raio': raio,
#                            'faixa': faixa,
#                            'limite_inferior': limite_inferior,
#                            'limite_superior': limite_superior,
#                            'proporcao': proporcao
#                        })
#
#    return pd.DataFrame(resultados)

In [148]:
import pandas as pd

def calcular_heterofilia(census_centros_gdf, raios, faixas):
    resultados = []

    for raio in raios:
        for faixa, (limite_inferior, limite_superior) in faixas.items():
            for index, row in census_centros_gdf.iterrows():
                centro_id = row['GEOID']
                centro_referencia = row['centro']

                circulo = centro_referencia.buffer(raio)

                centros_dentro_raio = census_centros_gdf[census_centros_gdf.geometry.intersects(circulo)]
                total_dentro_raio = len(centros_dentro_raio)

                # Conta as celulas na região que esta dentro da faixa
                contagem_faixa = ((centros_dentro_raio['crime_number'] >= limite_inferior) & 
                                  (centros_dentro_raio['crime_number'] <= limite_superior)).sum()

                # Proporção de celulas na faixa pelo total da região
                if total_dentro_raio > 0:
                    proporcao = contagem_faixa / total_dentro_raio
                else:
                    proporcao = 0 

                resultados.append({
                    'centro_id': centro_id,
                    'raio': raio,
                    'faixa': faixa,
                    'limite_inferior': limite_inferior,
                    'limite_superior': limite_superior,
                    'proporcao': proporcao,
                    'total_dentro_raio': total_dentro_raio
                })

    return pd.DataFrame(resultados)

In [149]:
faixas = {
    'Faixa 1': (0, quartis[0.25]),
    'Faixa 2': (quartis[0.25], quartis[0.50]),
    'Faixa 3': (quartis[0.50], quartis[0.75]),
    'Faixa 4': (quartis[0.75], quartis[1])
}
faixas

{'Faixa 1': (0, 4931.0),
 'Faixa 2': (4931.0, 8146.0),
 'Faixa 3': (8146.0, 12878.0),
 'Faixa 4': (12878.0, 89599.0)}

In [150]:
raios = [2000, 5000, 10000]


vetor_heterofilia = calcular_heterofilia(census_centros_gdf, raios, faixas)
vetor_heterofilia.to_csv('heterofilia_chicago.csv', index=False)

print(vetor_heterofilia)

        centro_id   raio    faixa  limite_inferior  limite_superior  \
0     17031842400   2000  Faixa 1              0.0           4931.0   
1     17031840300   2000  Faixa 1              0.0           4931.0   
2     17031841100   2000  Faixa 1              0.0           4931.0   
3     17031841200   2000  Faixa 1              0.0           4931.0   
4     17031839000   2000  Faixa 1              0.0           4931.0   
...           ...    ...      ...              ...              ...   
9607  17031070400  10000  Faixa 4          12878.0          89599.0   
9608  17031070500  10000  Faixa 4          12878.0          89599.0   
9609  17031130300  10000  Faixa 4          12878.0          89599.0   
9610  17031292200  10000  Faixa 4          12878.0          89599.0   
9611  17031630900  10000  Faixa 4          12878.0          89599.0   

      proporcao  total_dentro_raio  
0      0.000000                  6  
1      0.142857                  7  
2      0.545455                 11  

In [151]:
centro25 = vetor_heterofilia[(vetor_heterofilia['centro_id'] == 17031611800) & (vetor_heterofilia['faixa'] == "Faixa 1")]
centro50 = vetor_heterofilia[(vetor_heterofilia['centro_id'] == 17031611800) & (vetor_heterofilia['faixa'] == "Faixa 2")]
centro75 = vetor_heterofilia[(vetor_heterofilia['centro_id'] == 17031611800) & (vetor_heterofilia['faixa'] == "Faixa 3")]
centro100 = vetor_heterofilia[(vetor_heterofilia['centro_id'] == 17031611800) & (vetor_heterofilia['faixa'] == "Faixa 4")]
print(centro25['proporcao'].values)
print(centro50['proporcao'].values)
print(centro75['proporcao'].values)


[0.         0.05633803 0.18326693]
[0.22222222 0.15492958 0.20318725]
[0.61111111 0.4084507  0.31075697]


In [152]:
estatisticas = vetor_heterofilia.groupby(['raio', 'faixa']).agg(
    media=('proporcao', 'mean'),
    desvio_padrao=('proporcao', 'std'),
    mediana=('proporcao', 'median'),
    count=('proporcao', 'size')
).reset_index()

print(estatisticas)

     raio    faixa     media  desvio_padrao   mediana  count
0    2000  Faixa 1  0.247752       0.258031  0.176471    801
1    2000  Faixa 2  0.245959       0.181627  0.230769    801
2    2000  Faixa 3  0.249132       0.175564  0.230769    801
3    2000  Faixa 4  0.240240       0.268519  0.125000    801
4    5000  Faixa 1  0.241231       0.180373  0.217391    801
5    5000  Faixa 2  0.243730       0.097950  0.238095    801
6    5000  Faixa 3  0.250901       0.094174  0.253521    801
7    5000  Faixa 4  0.247831       0.195906  0.200000    801
8   10000  Faixa 1  0.237297       0.115948  0.203390    801
9   10000  Faixa 2  0.248549       0.061557  0.252459    801
10  10000  Faixa 3  0.251468       0.053304  0.244526    801
11  10000  Faixa 4  0.246452       0.129841  0.234421    801


In [166]:
# Acessando os valores da coluna 'proporcao'
centros = {
    'centro25': centro25['proporcao'].values,
    'centro50': centro50['proporcao'].values,
    'centro75': centro75['proporcao'].values
}
centros

{'centro25': array([0.        , 0.05633803, 0.18326693]),
 'centro50': array([0.22222222, 0.15492958, 0.20318725]),
 'centro75': array([0.61111111, 0.4084507 , 0.31075697])}

In [176]:
import numpy as np
import ot

def wasserstein_dist(q1, q2):

    d_cpu = np.abs(np.subtract.outer(q1, q2))

    #print(d_cpu)

    return ot.emd2(q1, q2, d_cpu)

In [177]:
from ot.unbalanced import sinkhorn_unbalanced

def wasserstein_dist(q1, q2):

    q1 = q1 / np.sum(q1)
    q2 = q2 / np.sum(q2)

    M = ot.dist(q1, q2, metric='euclidean')

    print(M)
    print("\n")
    return ot.emd2(q1, q2, M)

In [178]:
# Calculando as distâncias automaticament
for key1 in centros:
    for key2 in centros:
        if key1 != key2:
            wd = wasserstein_dist(centros[key1], centros[key2])
            #print(f"q1  {centros[key1]} e q2 {centros[key2]}")
            print(f"Wasserstein Distance entre {key1} e {key2}:\n {wd} \n")

ValueError: einstein sum subscripts string contains too many subscripts for operand 0

In [160]:
# Calculando as distâncias automaticament
for key1 in centros:
    for key2 in centros:
        if key1 != key2:
            wd = wasserstein_dist(centros[key1], centros[key2])
            #print(f"q1  {centros[key1]} e q2 {centros[key2]}")
            print(f"Wasserstein Distance entre {key1} e {key2}: {wd} \n")

AssertionError: 
Arrays are not almost equal to 6 decimals
a and b vector must have the same sum
Mismatched elements: 1 / 1 (100%)
Max absolute difference: 0.34073409
Max relative difference: 0.58712935
 x: array(0.239605)
 y: array([0.580339])

In [156]:
#   import ot
#   import numpy as np
#   from scipy.special import kl_div
#   
#   def wasserstein_dist(q1, q2):
#   	similarity = js_div(q1, q2).sum(axis=-1)
#   	d_cpu = similarity.reshape(q1.shape[0], q2.shape[0])
#   
#   	return ot.emd2(q1, q2, d_cpu)
#   	
#   def js_div(q1, q2, get_softmax=True):
#   	log_mean_output = np.log((q1 + q2) / 2)
#   	return (kl_div(log_mean_output, q1) + kl_div(log_mean_output, q2)) / 2